# Audio Processor #
### Spring 2026 ###
### Final Project for Gustave Rohde, TuTh 2:00pm ###
### Created By: ###
* Matthew Bach
* Steve Philipose
* Ishmam 

[Github Repository](https://github.com/Littleamish2/audio_processor_effects.git)


### Architecture: ###
1. Input: audio input (real time stream or recorded file), filter parameters
2. Apply filters to audio input 
3. Output: Filtered audio 

### Effects: ###
* Equalization (Matthew)
* Reverb (Ishmam)
* Delay (Steve)
* Chorus Modulation (Matthew)
* Phaser modulation (Ishmam)
* Flanger Modulation (Steve)

In [30]:
import math 
from matplotlib import pyplot as plt 
import numpy as np 
from time import time 
import scipy
from scipy.io import wavfile
from IPython.display import display, Audio


# Input #

WAV File Processing

In [31]:
## Read WAV File
## Output Sampling Frequency & Mono Signal
def read_wav(fname):
    # Read the wav file given by fname and return the sampling frequency (fs) and
    # the audio data (mono_signal) as a np array
    fs, data = wavfile.read(fname)

    # Convert the data back to floating point values ranging from -1.0 to 1.0
    normalized_data = data / np.iinfo(np.int16).max

    if data.shape[1] > 1: # Covert Stereo to mono
        mono_signal = np.mean(normalized_data, axis=1)

    return fs, mono_signal

Fourier Transform for signal analysis

In [32]:
def time_to_freq(y, samplerate):
  # Returns the fourier transform of a signal
  # as well as the corresponding frequencies
  n = len(y)
  Y_full = np.fft.fft(y)
  freq_full = np.fft.fftfreq(n, d=1/samplerate)
  return Y_full, freq_full

def freq_to_time(Y):
  # Reconstructs the signal
  y = np.fft.ifft(Y)
  y = np.real(y)
  return y

def plot_freq(Y, freqs):
  # Plot only the magnitude of the FT for only positive frequencies
  # (ignore complex values and negative frequencies)
  plt.figure(figsize=(2.5,2.5))
  inds = freqs >=0
  plt.plot(freqs[inds], np.absolute(Y[inds]))
  plt.xlabel("Frequency")
  plt.ylabel("Fourier Transform Magnitude")

# Effects #

In [33]:
## Equalizer Effect

from enum import Enum, auto

class Curve(Enum):

    FLAT = auto()           ## Flat gain across all frequencies
    SMILEY_FACE = auto()    ## Based on equal-loudness curve 
    FROWNIE_FACE = auto()   ## Opposite of equal-loudness curve
    HIGH_FREQ = auto()      ## Applies only to high frequencies
    LOW_FREQ = auto()       ## Applies only to low frequencies

HIGH_FREQUENCY = 10000
LOW_FREQUENCY = 100


def equalizer(signal, sample_frequency, gain=1.0, curve_type=Curve.FLAT):
    freq_signal, freqs = time_to_freq(signal, sample_frequency)

    match curve_type:
        case Curve.FLAT:
            freq_signal *= gain
            return freq_to_time(freq_signal)
        case Curve.SMILEY_FACE: 
            ## Curve Based On a Cosine-Wave 
            ## Wave with f = f max (one full cycle)
            ## Scoop from gain (on low/high frequencies) to 1 (on mid frequencies)
            
            amplitude   = (gain-1)/2
            dc          = amplitude + 1
            
            freq_signal *= dc + amplitude * np.cos(2 * np.pi * freqs / max(freqs)) 

            return freq_to_time(freq_signal)
        
        case Curve.FROWNIE_FACE:
            ## Same as Smiley Face but flip the cosine wave (opposite phase)
            ## Scoop from 1 (on low/high frequencies) to gain (on mid frequencies)

            amplitude   = (gain-1)/2 
            dc          = amplitude + 1
            freq_signal *= dc + amplitude * -1 * np.cos(2 * np.pi * freqs / max(freqs)) 

            return freq_to_time(freq_signal)
        
        case Curve.HIGH_FREQ:
            return freq_to_time(freq_signal[freqs > HIGH_FREQUENCY] * gain)
        case Curve.LOW_FREQ:
            return freq_to_time(freq_signal[freqs < LOW_FREQUENCY] * gain)

    return signal

Reverb Effect

Schroeder Reverb: N parallel feedback comb filters summed and passed through K series all-pass filters

**Comb Filter (parallel):**
$$ y[n] = x[n] + g \cdot y[n-M] $$
$$ H(z) = \frac{1}{1 - g \cdot z^{-M}} $$

**All-Pass Filter (series, for diffusion):**
$$ y[n] = -g \cdot x[n] + x[n-M] + g \cdot y[n-M] $$
$$ H(z) = \frac{z^{-M} - g}{1 - g \cdot z^{-M}} $$

* $x[n]$ is the dry input signal
* $y[n]$ is the wet output signal
* $g$ is the feedback gain — controls decay length / room size
* $M$ is the delay in samples for each filter stage
* Delay times are chosen as mutually prime-ish values to minimise flutter echoes


In [34]:
def reverb(signal, sample_frequency, room_size=0.5, wet_level=0.33, dry_level=0.7):
    # Schroeder reverb: parallel feedback comb filters + series all-pass diffusers
    comb_delay_times   = [0.0297, 0.0371, 0.0411, 0.0437]  # ~30-44 ms, prime-ish
    allpass_delay_times = [0.005, 0.0017]

    feedback = 0.5 + room_size * 0.4  # room_size [0,1] maps to gain [0.5, 0.9]

    def comb_filter(x, dt, g):
        M = int(dt * sample_frequency)
        b = np.zeros(M + 1); b[0] = 1.0
        a = np.zeros(M + 1); a[0] = 1.0; a[M] = -g
        return scipy.signal.lfilter(b, a, x)

    def allpass_filter(x, dt, g=0.5):
        M = int(dt * sample_frequency)
        b = np.zeros(M + 1); b[0] = -g;  b[M] = 1.0
        a = np.zeros(M + 1); a[0] = 1.0; a[M] = -g
        return scipy.signal.lfilter(b, a, x)

    wet = sum(comb_filter(signal, dt, feedback) for dt in comb_delay_times) / len(comb_delay_times)

    for dt in allpass_delay_times:
        wet = allpass_filter(wet, dt)

    return dry_level * signal + wet_level * wet


Delay Effect

$$ y[n] = x[n] + gain \cdot x[n-N] $$
* $x[n]$ is the "dry" input signal
* $y[n]$ is the "wet" output signal
* $gain$ is the gain of the delayed signal
* $N$ is the delay in samples, calculated as $delay\_time \cdot f_s$


In [35]:
def delay(signal, sample_frequency, delay_time=0.5, gain=0.5):
    delay_samples = int(delay_time * sample_frequency)
    delayed_signal = np.zeros_like(signal)
    if delay_samples < len(signal):
        delayed_signal[delay_samples:] = signal[:-delay_samples]
    return signal + gain * delayed_signal


Chorus Modulation

$$ y[n] = x[n] + gain \cdot x[n-D[n]] $$
* $x[n]$ is the "dry" input signal
* $y[n]$ is the "wet" output signal
* $g$ is the gain of the delayed signal

$$ D[n] = M + D_{depth} \cdot \sin (2\pi f_{LFO} \cdot n/f_s)$$
* $ D[n] $ is the time-varying delay
* $M$ is the mean delay time (10-30 ms)
* $D_{depth}$ is the modulation depth / variation in delay time
* $f_{LFO}$ is the low frequency oscillator's frequency
* $f_s$ is the sample rate


In [36]:
def chorusMod(signal, sample_frequency, gain=0.5, modulation_frequency=1.5, modulation_depth=0.002, mean_delay_time=0.003):
    n = np.arange(len(signal))

    # Delay in seconds
    d = mean_delay_time + modulation_depth * np.sin(2 * np.pi * modulation_frequency * n / sample_frequency)

    # Convert to samples
    d_samples = (d * sample_frequency).astype(int)

    y = np.zeros_like(signal)

    N = len(signal)

    for i in range(N):
        idx = i - d_samples[i]

        if 0 <= idx < N:
            y[i] = signal[i] + gain * signal[idx]
        else:
            y[i] = signal[i]

    return y

Phaser Modulation

Cascaded first-order all-pass filters whose cutoff frequency is swept by an LFO.
Mixing the phase-shifted (wet) signal with the original (dry) creates sweeping notches.

**First-Order All-Pass Filter:**
$$ H(z) = \frac{g + z^{-1}}{1 + g \cdot z^{-1}}, \quad |H(e^{j\omega})| = 1 \; \forall \omega $$

Time domain: $ y[n] = g[n] \cdot x[n] + x[n-1] - g[n] \cdot y[n-1] $

**Time-Varying Coefficient (Bilinear Transform):**
$$ g[n] = \frac{\tan(\pi f_c[n] / f_s) - 1}{\tan(\pi f_c[n] / f_s) + 1} $$

**LFO-Swept Cutoff Frequency:**
$$ f_c[n] = f_{center} \cdot \left(1 + D \cdot \sin\left(2\pi f_{LFO} \cdot \frac{n}{f_s}\right)\right) $$

* $f_{LFO}$ is the LFO rate (Hz)
* $D$ is the modulation depth (0–1)
* $f_{center}$ is the center sweep frequency (Hz)
* num\_stages: number of cascaded all-pass sections (typically 2, 4, 6, or 8)


In [37]:
def phaserMod(signal, sample_frequency, num_stages=4, lfo_rate=0.5,
              lfo_depth=0.7, center_freq=1000.0, wet_dry=0.5):
    n   = np.arange(len(signal))
    lfo = 0.5 * (1 + np.sin(2 * np.pi * lfo_rate * n / sample_frequency))  # 0→1

    # LFO sweeps the all-pass cutoff frequency around center_freq
    f_sweep = center_freq * (1 - lfo_depth + 2 * lfo_depth * lfo)
    f_sweep = np.clip(f_sweep, 20.0, sample_frequency / 2.0 - 100.0)

    # Bilinear transform: time-varying all-pass coefficient g[n]
    t = np.tan(np.pi * f_sweep / sample_frequency)
    g = (t - 1.0) / (t + 1.0)

    # Cascaded time-varying first-order all-pass filters
    # y[n] = g[n]*x[n] + x[n-1] - g[n]*y[n-1]
    x = signal.copy()
    for _ in range(num_stages):
        y = np.zeros(len(signal))
        for i in range(1, len(signal)):
            y[i] = g[i] * x[i] + x[i - 1] - g[i] * y[i - 1]
        x = y

    return (1 - wet_dry) * signal + wet_dry * x


Flanger Modulation

$$ y[n] = x[n] + gain \cdot x[n-D[n]] $$
* $x[n]$ is the "dry" input signal
* $y[n]$ is the "wet" output signal
* $gain$ is the gain of the delayed signal

$$ D[n] = M + D_{depth} \cdot \sin (2\pi f_{LFO} \cdot n/f_s)$$
* $ D[n] $ is the time-varying delay
* $M$ is the mean delay time (1-10 ms)
* $D_{depth}$ is the modulation depth / variation in delay time
* $f_{LFO}$ is the low frequency oscillator's frequency
* $f_s$ is the sample rate


In [38]:
def flangerMod(signal, sample_frequency, gain=0.5, modulation_frequency=1.0, modulation_depth=0.002, mean_delay_time=0.003):
    n = np.arange(len(signal))
    delay_time_sec = mean_delay_time + modulation_depth * np.sin(2 * np.pi * modulation_frequency * n / sample_frequency)
    delay_samples = np.int32(delay_time_sec * sample_frequency)
    
    indices = n - delay_samples
    valid = indices >= 0
    
    y = np.copy(signal)
    y[valid] += gain * signal[indices[valid]]
    return y


# Output #


WAV File Processing

In [39]:
# Write WAV File (fileName, signal, sampling frequency)
def write_wav(fname, sig, fs):
  scaled = np.int16(sig / np.max(np.abs(sig)) * np.iinfo(np.int16).max)
  wavfile.write(fname, fs, scaled)

# Audio Processor #

In [40]:
## Pre-packaged inputs in InputAudio folder
ELECTRIC_GUITAR = "electric-guitar-melody.wav"
ACOUSTIC_GUITAR = "loud-acoustic-happy.wav"
ONE_SHOT_STRING = "string-one-shot-guitar-clean.wav"


chosen_input = ELECTRIC_GUITAR 

In [41]:
## Read Input WAV File
input_file = open(f"./InputAudio/{chosen_input}", "r")
fs, mono_signal = read_wav(input_file.name)
input_file.close()

In [42]:
## Apply Audio Effects
equalized_signal = equalizer(mono_signal, fs)
reverb_signal = reverb(mono_signal, fs)
delay_signal = delay(mono_signal, fs)
chorus_signal = chorusMod(mono_signal, fs)
phaser_signal = phaserMod(mono_signal, fs)
flanger_signal = flangerMod(mono_signal, fs)

all_in_one_signal = flangerMod(chorusMod(phaserMod(delay(reverb(equalizer(mono_signal, fs), fs), fs), fs), fs), fs)

In [43]:
## Write Output WAV Files
write_wav(f"./OutputAudio/equalized_signal_{chosen_input}.wav", equalized_signal, fs)
write_wav(f"./OutputAudio/reverb_signal_{chosen_input}.wav", reverb_signal, fs)
write_wav(f"./OutputAudio/delay_signal_{chosen_input}.wav", delay_signal, fs)
write_wav(f"./OutputAudio/chorus_signal_{chosen_input}.wav", chorus_signal, fs)
write_wav(f"./OutputAudio/phaser_signal_{chosen_input}.wav", phaser_signal, fs)
write_wav(f"./OutputAudio/flanger_signal_{chosen_input}.wav", flanger_signal, fs)
write_wav(f"./OutputAudio/all_in_one_signal_{chosen_input}.wav", all_in_one_signal, fs)

In [44]:
## Display Audio for Listening
display(Audio(equalized_signal, rate=fs))
display(Audio(reverb_signal, rate=fs))
display(Audio(delay_signal, rate=fs))
display(Audio(chorus_signal, rate=fs))
display(Audio(phaser_signal, rate=fs))
display(Audio(flanger_signal, rate=fs))
display(Audio(all_in_one_signal, rate=fs))